In [1]:
import pandas as pd
import numpy as np

In [2]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

In [4]:
df = pd.read_csv("covid_toy.csv")
df.sample(5)

,age,gender,fever,cough,city,has_covid
86,25,Male,104.0,Mild,Bangalore,Yes
26,19,Female,100.0,Mild,Kolkata,Yes
68,54,Female,104.0,Strong,Kolkata,No
54,60,Female,99.0,Mild,Mumbai,Yes
71,75,Female,104.0,Strong,Delhi,No


In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df.drop("has_covid", axis=1), df["has_covid"], test_size=0.2)

In [14]:
# Hectic Way

# Simple Imputer
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[["fever"]])
X_test_fever = si.fit_transform(X_test[["fever"]])

X_train_fever.shape

(80, 1)

In [16]:
# Ordinal Encoding

oe = OrdinalEncoder(categories=[["Mild", "Strong"]])

X_train_cough = oe.fit_transform(X_train[["cough"]])
X_test_cough = oe.fit_transform(X_test[["cough"]])

X_train_cough.shape

(80, 1)

In [25]:
# One Hot Encoding

ohe = OneHotEncoder(drop="first")

X_train_gender_city = ohe.fit_transform(X_train[["gender", "city"]])
X_train_gender_cityX_test__gender_city = ohe.fit_transform(X_test[["gender", "city"]])

X_train_gender_city.shape

(80, 4)

In [27]:
# Extracting Age

X_train_age = X_train.drop(columns=["gender", "city", "fever", "cough"]).values
X_test_age = X_test.drop(columns=["gender", "city", "fever", "cough"]).values

X_train_age.shape


(80, 1)

In [28]:
# Using Column Transformer
from sklearn.compose import ColumnTransformer
transformer = ColumnTransformer(transformers=[
    ("tnf1", SimpleImputer(), ["fever"]), 
    ("tnf2", OrdinalEncoder(categories=[["Mild", "Strong"]]), ["cough"]),
    ("tnf3", OneHotEncoder(drop="first"), ["gender", "city"])
], remainder="passthrough")

In [30]:
transformer.fit_transform(X_train).shape

(80, 7)

In [39]:
flats = transformer.named_transformers_["tnf3"].get_feature_names_out(
    ["gender", "city"]
)

In [49]:
vals = transformer.transform(X_train)
vals.shape
", ".join(flats)

'gender_Male, city_Delhi, city_Kolkata, city_Mumbai'

In [50]:
df = pd.DataFrame(vals, columns=["fever", "cough",  *flats, "age"],)
# ["fever", "cough", ", ".join(flats), "age"]

In [51]:
df

,fever,cough,gender_Male,city_Delhi,city_Kolkata,city_Mumbai,age
0,101.0,1.0,1.0,0.0,0.0,0.0,47.0
1,101.0,0.0,0.0,0.0,0.0,1.0,81.0
2,104.0,0.0,1.0,0.0,0.0,1.0,42.0
3,101.0,1.0,0.0,1.0,0.0,0.0,68.0
4,104.0,0.0,0.0,0.0,1.0,0.0,17.0
...,...,...,...,...,...,...,...
75,101.0,0.0,1.0,1.0,0.0,0.0,15.0
76,104.0,0.0,1.0,0.0,0.0,0.0,25.0
77,100.0,1.0,0.0,0.0,0.0,0.0,19.0
78,104.0,1.0,0.0,1.0,0.0,0.0,75.0
